# 03 · Filter & Rank — the shared enzyme filter (metal-site geometry)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 20** you filter on the **enzyme** cutoffs, with the **metal-ligand
geometry** (mapped onto `catalytic_geom_rmsd`) as the decisive metric, plus a solubility check and a
GRACE-style **CLEAN-style functional classification**.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
Improvements here are pull-requested back to `shared/` for the whole cohort — do not silently fork it.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("enzyme cutoffs:", fp.DEFAULT_CUTOFFS["enzyme"])
print("(catalytic_geom_rmsd <= 0.5 is filled here by the METAL-LIGAND RMSD)")

## Build `fp.Design` objects (enzyme) from the campaign
Map each campaign row onto an `fp.Design`, carrying the enzyme-specific fields: `plddt`,
**`plddt_catalytic`**, `scrmsd`, **`catalytic_geom_rmsd`** (= the **metal-ligand RMSD**), `solubility`,
and `md_rmsd`. The self-consistency layer checks these against the enzyme cutoffs (scrmsd ≤ 2.0,
plddt ≥ 85, plddt_cat ≥ 90, cat_geom ≤ 0.5); the physics layer checks solubility.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence=str(r.get("sequence", "")),
        design_type="enzyme",
        plddt=float(r["plddt"]),
        plddt_catalytic=float(r["plddt_catalytic"]),
        scrmsd=float(r["scrmsd"]),
        catalytic_geom_rmsd=float(r["metal_ligand_rmsd"]),   # METAL-LIGAND geometry is the key metric
        solubility=float(r["solubility"]),
        md_rmsd=float(r["md_rmsd"]),
        extra={"scaffold_method": r["scaffold_method"], "design_tool": r["design_tool"],
               "metal_aware": bool(r["metal_aware"]), "synthetic": True},
    ))
print(len(designs), "enzyme Design objects built (from SYNTHETIC mock metrics)")

## Run the pipeline (`design_type="enzyme"`) and report
`run_pipeline` applies the layers in order and returns a ranked DataFrame. We use layers 1+3+4
(self-consistency incl. metal-ligand geometry, physics incl. solubility, and the caveated short-MD
dynamics layer); the orthogonal layer (L2) needs a second predictor's scRMSD, which you add on Colab.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="enzyme", use_layers=(1, 3, 4))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj20")
top

## GRACE-style functional triage: a CLEAN-style classification (mock)
GRACE pairs geometry with a **functional classifier** (CLEAN-style EC/function prediction) + solubility
to prune a large pool. The real call runs CLEAN on each sequence; here a deterministic mock flags each
design as the intended class or not, so the plumbing runs. **SYNTHETIC** — never report as real.

In [ ]:
import hashlib

def clean_style_classification_mock(seq, design_id):
    """SYNTHETIC stand-in for a CLEAN-style EC/functional classifier (verify + wire up the real tool).
    Returns (predicted_class, confidence). On Colab: run CLEAN and parse its EC prediction."""
    h = int(hashlib.sha256(("clean" + str(design_id)).encode()).hexdigest(), 16)
    is_ca = (h % 100) < 65          # SYNTHETIC: ~65% read as the intended class
    conf = round(0.5 + (h % 50) / 100.0, 2)
    return ("carbonic-anhydrase-like (EC 4.2.1.1)" if is_ca else "other/uncertain"), conf

camp["clean_class"], camp["clean_conf"] = zip(*[
    clean_style_classification_mock(s, i) for s, i in zip(camp["sequence"], camp["design_id"])])
camp.to_csv("results/campaign.csv", index=False)
n_ca = (camp["clean_class"].str.startswith("carbonic")).sum()
print(f"CLEAN-style classification [SYNTHETIC]: {n_ca}/{len(camp)} read as carbonic-anhydrase-like")
print("On Colab: replace with the real CLEAN EC prediction; pair it with solubility to prune the pool.")

## Survival-at-each-layer + metal-geometry pass rate (honest accounting)
The **metal-geometry layer is where most metalloenzyme designs die** — expect the steepest drop there.
Report the pass rate explicitly; this is a headline benchmark for D3.

In [ ]:
print("layers_passed distribution:")
print(df_ranked["layers_passed"].value_counts().sort_index())

n = len(df_ranked)
cut = fp.DEFAULT_CUTOFFS["enzyme"]["cat_geom"]
n_geom = int((df_ranked["catalytic_geom_rmsd"] <= cut).sum())
print(f"\nmetal-geometry preservation: {n_geom}/{n} "
      f"({100*n_geom/max(n,1):.1f}%) hold the Zn-His3 cage < {cut} A  [SYNTHETIC demo numbers]")
print("Reminder: geometry != metal incorporation != activity. ICP + an assay decide (notebook 05).")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="enzyme"`.
- [ ] Survival-at-each-layer figure (`results/proj20_survival.png`).
- [ ] Metal-geometry preservation rate reported (the headline metric).
- [ ] Solubility + CLEAN-style functional classification added (GRACE-style triage).
- [ ] Mapping assumptions written down (metal-ligand RMSD → `catalytic_geom_rmsd`; AF2 doesn't place the Zn).

**Next:** `04_validate.ipynb` — metal-geometry + LigandMPNN-vs-ProteinMPNN + pool-size-vs-hit-rate + docking/MD.